# 05 — Clustering and classification
Research question: What starting-position profiles and theoretical resource advantages can be identified before the first dice roll?

Part A is unsupervised clustering. Part B is supervised classification of an analytical proxy, NOT a real game outcome. No game is played or rolled; only the mathematical distribution of two dice is used. The proxy asks whether a fixed initial position has at least a 10% probability of accumulating the resource cost of one road plus one settlement in 20 hypothetical independent rolls, ignoring trading, robber moves, bank shortages, other actions and spending. The horizon and threshold are exploratory choices; a pilot of 8 rolls and 25% produced no positive examples. This proxy does not test whether a legal settlement is built, whether a player wins, or optimal strategy. The exact probability is preferable to a classifier if full board information is available.

In [ ]:
from pathlib import Path
from importlib import import_module
from collections import defaultdict
from itertools import product
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score, ConfusionMatrixDisplay
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.dummy import DummyClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

OUT = Path("processed_data")
p = pd.read_csv(OUT / "initial_placements.csv")
t = pd.read_csv(OUT / "map_tiles.csv")
f = pd.read_csv(OUT / "initial_features.csv")
fe = import_module("03_feature_engineering")
nodes, edges, neighbors = fe.build_topology(t)
resources = ["WOOD","BRICK","SHEEP","WHEAT","ORE"]
cols = [r.title()+"_Production" for r in resources] + ["Number_Diversity","Expansion_Options"] + ["Initial_"+r.title() for r in resources]
# No identifiers, raw node IDs, target, or exact aggregate duplicates.
assert not f[cols].isna().any().any()

## A. Unsupervised learning
Standardize numerical features; compare K-Means and Ward hierarchical clustering for k = 2–6 with silhouette scores. PCA is used for visualization, not for fitting the clusters.

In [ ]:
z = StandardScaler().fit_transform(f[cols])
cluster_results, fitted_labels = [], {}
for k in range(2,7):
    for name, estimator in [
        ("KMeans",KMeans(n_clusters=k,n_init=20,random_state=42)),
        ("Ward",AgglomerativeClustering(n_clusters=k))]:
        labels = estimator.fit_predict(z)
        cluster_results.append({"Method":name,"K":k,"Silhouette":silhouette_score(z,labels)})
        fitted_labels[name,k] = labels
cluster_scores = pd.DataFrame(cluster_results).sort_values("Silhouette",ascending=False)
display(cluster_scores)
best = cluster_scores.iloc[0]
labels = fitted_labels[best.Method,int(best.K)]
profiles = f.assign(Cluster=labels).groupby("Cluster")
display(profiles[cols+["Total_Production","Resource_Diversity","Initial_Total"]].mean().round(2))
display(profiles.size().rename("Count"))
xy = PCA(n_components=2).fit_transform(z)
plt.scatter(xy[:,0],xy[:,1],c=labels,cmap="tab10",alpha=.7)
plt.xlabel("PC1"); plt.ylabel("PC2"); plt.title("Starting-position clusters (PCA)")
plt.colorbar(label="Cluster"); plt.show()
cluster_scores.to_csv(OUT / "cluster_scores.csv",index=False)
f.assign(Cluster=labels).to_csv(OUT / "clustered_positions.csv",index=False)

## B. Calculate the analytical label
The resource cost vector is (2 wood, 2 brick, 1 sheep, 1 wheat). Dynamic programming tracks only quantities up to those requirements. Simultaneous production from the same dice number is preserved. Every roll value is integrated over exactly; no random roll is generated. Seven yields zero resources.

In [ ]:
required = ["WOOD","BRICK","SHEEP","WHEAT"]
caps = (2,2,1,1)
states = list(product(*(range(c+1) for c in caps)))
idx = {s:i for i,s in enumerate(states)}
HORIZON = 20
THRESHOLD = 0.10

def readiness(start, gains, horizon=HORIZON):
    v = np.zeros(len(states))
    v[idx[tuple(min(x,c) for x,c in zip(start,caps))]] = 1
    for _ in range(horizon):
        nv = np.zeros_like(v)
        for n in range(2,13):
            weight = (6-abs(7-n))/36
            for state,i in idx.items():
                target = tuple(min(c,x+y) for c,x,y in zip(caps,state,gains[n]))
                nv[idx[target]] += weight*v[i]
        v = nv
    assert np.isclose(v.sum(),1)
    return v[idx[caps]]

rows = []
for gid,g in p.groupby("Game_ID"):
    board = t[t.Game_ID==gid].set_index("Tile_ID")
    for r in g.itertuples(index=False):
        start = [0]*4
        gains = defaultdict(lambda:[0]*4)
        for j,node in enumerate((r.Settlement_1,r.Settlement_2)):
            for tid in nodes[node]:
                tile = board.loc[tid]
                if tile.Resource not in required:
                    continue
                k = required.index(tile.Resource)
                gains[int(tile.Number)][k] += 1
                if j == 1:
                    start[k] += 1
        rows.append((gid,r.Player,readiness(start,gains)))
proxy = pd.DataFrame(rows,columns=["Game_ID","Player","Readiness_Probability"])
d = f.merge(proxy,on=["Game_ID","Player"],validate="one_to_one")
d["Class"] = (d.Readiness_Probability >= THRESHOLD).astype(int)
assert len(d)==len(f) and d.Readiness_Probability.between(0,1).all()
display(d.Readiness_Probability.describe())
display(d.Class.value_counts())
print("Zero probability:",d.Readiness_Probability.eq(0).sum())
d.to_csv(OUT / "labeled_positions.csv",index=False)

## C. Compare supervised algorithms
Use a group-based 80/20 split to prevent players from the same board appearing in train and test sets. Report balanced accuracy, F1 for the minority class and ROC-AUC in addition to accuracy. Scaling is fitted on training data only. Five-fold stratified group CV is a separate stability check; no hyperparameter search or external test set is claimed.

In [ ]:
X, y = d[cols], d.Class
train,test = next(GroupShuffleSplit(n_splits=1,test_size=.2,random_state=42).split(X,y,d.Game_ID))
assert not set(d.Game_ID.iloc[train]) & set(d.Game_ID.iloc[test])
assert y.iloc[train].nunique()==y.iloc[test].nunique()==2
print("Train:",y.iloc[train].value_counts().to_dict(),"Test:",y.iloc[test].value_counts().to_dict())
models = {
    "Dummy":DummyClassifier(strategy="most_frequent"),
    "KNN":make_pipeline(StandardScaler(),KNeighborsClassifier(n_neighbors=7)),
    "NB":GaussianNB(),
    "CART":DecisionTreeClassifier(max_depth=4,min_samples_leaf=8,random_state=42),
    "RF":RandomForestClassifier(n_estimators=200,min_samples_leaf=3,random_state=42),
    "LR":make_pipeline(StandardScaler(),LogisticRegression(max_iter=1000,random_state=42))
}
results = []
for name,model in models.items():
    model.fit(X.iloc[train],y.iloc[train])
    pred = model.predict(X.iloc[test])
    proba = model.predict_proba(X.iloc[test])[:,1]
    results.append({"Model":name,"Accuracy":accuracy_score(y.iloc[test],pred),
        "Balanced_Accuracy":balanced_accuracy_score(y.iloc[test],pred),
        "F1":f1_score(y.iloc[test],pred,zero_division=0),
        "ROC_AUC":roc_auc_score(y.iloc[test],proba)})
    ConfusionMatrixDisplay.from_predictions(y.iloc[test],pred)
    plt.title(name); plt.show()
results = pd.DataFrame(results)
display(results.round(4))
results.to_csv(OUT / "classification_results.csv",index=False)

In [ ]:
cv = StratifiedGroupKFold(n_splits=5,shuffle=True,random_state=42)
cv_rows = []
for name,model in models.items():
    scores = cross_validate(model,X,y,groups=d.Game_ID,cv=cv,scoring=["balanced_accuracy","f1","roc_auc"],error_score="raise")
    row = {"Model":name}
    for metric in ["balanced_accuracy","f1","roc_auc"]:
        row[metric+"_mean"] = scores["test_"+metric].mean()
        row[metric+"_std"] = scores["test_"+metric].std()
    cv_rows.append(row)
cv_results = pd.DataFrame(cv_rows)
display(cv_results.round(4))
cv_results.to_csv(OUT / "classification_cv.csv",index=False)

In [ ]:
plt.figure(figsize=(14,7))
plot_tree(models["CART"],feature_names=cols,class_names=["Low","High"],filled=True,fontsize=7)
plt.show()
pd.Series(models["RF"].feature_importances_,index=cols).sort_values().plot.barh()
plt.title("Random Forest impurity importance (exploratory)")
plt.tight_layout(); plt.show()

Interpretation: a high score on this benchmark means the classifier approximates a deterministic, theoretical resource-readiness calculation from summary features. It does not validate actual starting advantage against human or strategic bots. A large number of zeros and weak cluster separation are important findings, not errors.